## ts analysis using access

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

In [ ]:
### Functions needed for the analysis

In [7]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    # ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False, cbar_orientation='vertical', hatch_type = 'insig'):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            if hatch_type == 'insig':
                pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            elif hatch_type == 'sig':
                pval_plot = np.ma.masked_greater(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = cbar_orientation, shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [8]:
from functions import preproc_funcs as funcs

In [9]:
from functions import xr_lowess

In [6]:
import glob

In [10]:
files_2030 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f2/Amon/uas/gn/*/*.nc'))
files_2035 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i2p1f2/Amon/uas/gn/*/*.nc'))
files_2040 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i3p1f2/Amon/uas/gn/*/*.nc'))
files_2045 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/uas/gn/*/*.nc'))
files_2050 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i5p1f2/Amon/uas/gn/*/*.nc'))
files_2055 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i6p1f2/Amon/uas/gn/*/*.nc'))
files_2060 = sorted(glob.glob('/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i7p1f2/Amon/uas/gn/*/*.nc'))
files_2045

['/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/uas/gn/v20250307/uas_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn9oPLVl3818947.nc',
 '/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/uas/gn/v20250307/uas_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_020101-030012.nc',
 '/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/uas/gn/v20250307/uas_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_030101-040012.nc',
 '/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/uas/gn/v20250307/uas_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_040101-050012.nc',
 '/scratch/p66/ars599/ACCESS_output/APP_output/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/uas/gn/v20250307/uas_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_050101-060012.nc',
 '/scratch/p66/ars599/ACCESS_output/APP_output

In [11]:
ds2030 = xr.open_mfdataset(files_2030)
ds2035 = xr.open_mfdataset(files_2035)
ds2040 = xr.open_mfdataset(files_2045)
ds2050 = xr.open_mfdataset(files_2050)
ds2055 = xr.open_mfdataset(files_2055)
ds2060 = xr.open_mfdataset(files_2060)

ValueError: All-NaN slice encountered

In [ ]:
ds2060